# Combine tracking to NFL performance: which movement measures hold up

NFL Big Data Bowl 2027. This notebook reproduces every number and figure in the write-up.

**What it does.** It tests whether movement measured by the 10 Hz Combine sensors predicts NFL regular-season
performance beyond the stopwatch and body size, for receivers, offensive linemen, defensive linemen and defensive
backs, and it measures how reliably each movement measure repeats. The result is mostly negative, and the notebook
shows why.

**How to run.** Attach the competition data, then use *Run All*. Cells must run in order because later cells reuse
earlier results. It needs `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib` and `duckdb`.
`duckdb` is used to read the large game-tracking files without loading them into memory; if it is missing, the
first cell tries to install it.

**What to expect.** Each section prints its results as text tables. Four figures are drawn at the end, and every
figure is saved with a CSV of the plotted values. All random steps use fixed seeds.

In [ ]:
import contextlib
import os
import subprocess
import sys
from pathlib import Path

try:
    import duckdb  # noqa: F401
except ImportError:  # only works if the notebook has internet access
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])

# Find the competition data: the folder that holds combine_tracking.csv.
CANDIDATES = [os.environ.get("NB_DATA_DIR", ""), "/kaggle/input"]
DATA_DIR = None
for root in CANDIDATES:
    if root and Path(root).exists():
        hits = list(Path(root).rglob("combine_tracking.csv"))
        if hits:
            DATA_DIR = hits[0].parent
            break
assert DATA_DIR is not None, "combine_tracking.csv not found: attach the competition data"
print("data folder:", DATA_DIR)

# Everything we write goes in the working folder: work/ (Parquet copies), results/ (text output), figures/.
HERE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
os.chdir(HERE)
for d in ("work", "results", "figures"):
    (HERE / d).mkdir(exist_ok=True)


class _Tee:
    """Print to the notebook and also save the output of a section in results/."""

    def __init__(self, name):
        self.f = open(HERE / "results" / f"{name}.txt", "w", encoding="utf8")
        self.old = sys.stdout
        sys.stdout = self

    def write(self, s):
        self.old.write(s)
        self.f.write(s)

    def flush(self):
        self.old.flush()

    def stop(self):
        sys.stdout = self.old
        self.f.close()

## 1. Load the data

Convert the contest CSVs to Parquet once, so later steps query them quickly.

Reads ../nfl-big-data-bowl-2027/*.csv, writes work/*.parquet.

In [ ]:
_log = _Tee('01_load')
sys.argv = ['notebook']
from pathlib import Path

import duckdb

SRC = DATA_DIR
OUT = HERE / "work"
OUT.mkdir(exist_ok=True)

FILES = [
    "combine_results", "combine_tracking", "players", "player_career_successes",
    "games", "player_play", "game_tracking_2023", "game_tracking_2024", "game_tracking_2025",
]

con = duckdb.connect()
for name in FILES:
    src = SRC / f"{name}.csv"
    dst = OUT / f"{name}.parquet"
    con.execute(
        f"COPY (SELECT * FROM read_csv('{src.as_posix()}', header=true, nullstr='NA', sample_size=-1)) "
        f"TO '{dst.as_posix()}' (FORMAT parquet)"
    )
    n = con.execute(f"SELECT count(*) FROM '{dst.as_posix()}'").fetchone()[0]
    print(f"{name}: {n} rows")

_log.stop()

## 2. Receiver and DB movement features (route drills)

Break features for the WR and DB drills, computed from smoothed positions. Speed and heading are recomputed because the file's `dis` and `s` columns disagree in scale (means 3.0 and 4.9 yd/s).

In [ ]:
_log = _Tee('02_route_features')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

DT = 0.1            # seconds per frame (10 Hz)
SMOOTH = 3          # frames in the centred moving average
TURN_WINDOW = 5     # frames over which a heading change is measured
MIN_SPEED = 2.0     # yd/s; the break must happen while moving
DRILL_TYPES = ("SKILL_DRILLS_WR", "SKILL_DRILLS_DB")


def _smooth(v: np.ndarray, k: int = SMOOTH) -> np.ndarray:
    pad = k // 2
    p = np.pad(v, pad, mode="edge")
    return np.convolve(p, np.ones(k) / k, mode="valid")


def _wrap(deg: np.ndarray) -> np.ndarray:
    return (deg + 180.0) % 360.0 - 180.0


def attempt_features(g: pd.DataFrame) -> dict | None:
    """One athlete, one attempt, frames in time order."""
    if len(g) < TURN_WINDOW + 6:
        return None
    x, y = _smooth(g["x"].to_numpy()), _smooth(g["y"].to_numpy())
    vx, vy = np.gradient(x, DT), np.gradient(y, DT)
    speed = np.hypot(vx, vy)
    heading = np.degrees(np.arctan2(vy, vx))
    turn = np.abs(_wrap(heading[TURN_WINDOW:] - heading[:-TURN_WINDOW]))
    moving = speed[:-TURN_WINDOW] >= MIN_SPEED
    if not moving.any():
        return None
    turn = np.where(moving, turn, 0.0)
    i = int(np.argmax(turn))             # window start; the break lies in [i, i + TURN_WINDOW]
    j = i + TURN_WINDOW
    entry = float(speed[max(0, i - 3):i + 1].mean())
    seg = speed[i:j + 1]
    min_speed = float(seg.min())
    k = i + int(seg.argmin())
    target = 0.9 * entry
    after = np.nonzero(speed[k:] >= target)[0]
    reaccel = float(after[0] * DT) if len(after) else np.nan
    return {
        "frames": len(g),
        "break_angle": float(turn[i]),
        "break_frame": i,
        "entry_speed": entry,
        "min_speed": min_speed,
        "speed_lost": entry - min_speed,
        "reaccel_s": reaccel,
        "peak_speed": float(speed.max()),
        "path_len": float(np.hypot(np.diff(x), np.diff(y)).sum()),
    }


def build(con: duckdb.DuckDBPyConnection) -> pd.DataFrame:
    df = con.execute(
        """
        SELECT draft_year, event_id, nfl_id, drill_type, drill_name, attempt, time, x, y
        FROM 'work/combine_tracking.parquet'
        WHERE entity_type = 'PLAYER' AND drill_type IN ('SKILL_DRILLS_WR', 'SKILL_DRILLS_DB')
        ORDER BY event_id, nfl_id, time
        """
    ).df()
    rows = []
    for (ev, nid), g in df.groupby(["event_id", "nfl_id"], sort=False):
        f = attempt_features(g)
        if f is None:
            continue
        first = g.iloc[0]
        rows.append({"event_id": ev, "nfl_id": nid, "draft_year": first.draft_year,
                     "drill_type": first.drill_type, "drill_name": first.drill_name,
                     "attempt": first.attempt, **f})
    return pd.DataFrame(rows)


if __name__ == "__main__":
    con = duckdb.connect()
    out = build(con)
    out.to_parquet(HERE / "work" / "route_features.parquet")
    print(len(out), "attempts")
    pd.set_option("display.width", 200)
    print(out.groupby("drill_name")[["break_angle", "entry_speed", "speed_lost", "reaccel_s"]]
          .median().round(2).to_string())

_log.stop()

## 3. Receiver outcomes: separation over expected

NFL receiver outcomes from player_play, one row per player (and per player and route).

Separation is only recorded when a pass was thrown, so it is a measure of getting open on
routes where the quarterback let go of the ball. It depends heavily on route type and on
man versus zone coverage, so each play's separation is compared with the average for the
same route and coverage type ("separation over expected"). Plays voided by a penalty are
excluded.

In [ ]:
_log = _Tee('03_wr_outcomes')
sys.argv = ['notebook']
from pathlib import Path

import duckdb

MIN_PLAYS = 30      # pass-forward plays a player needs for a stable average
MIN_ROUTE_PLAYS = 10

con = duckdb.connect()
con.execute("""
CREATE VIEW plays AS
SELECT pp.nfl_id, pl.nfl_position, pl.draft_year, pp.route_ran, pp.team_coverage_man_zone AS cov,
       pp.separation_at_pass_forward AS sep, pp.cushion, pp.target::INT AS target
FROM 'work/player_play.parquet' pp
JOIN 'work/players.parquet' pl USING (nfl_id)
WHERE pp.route_ran IS NOT NULL AND pp.play_nullified_by_penalty = 'N'
""")
con.execute("""
CREATE VIEW plays_x AS
SELECT p.*,
       p.sep - avg(p.sep) OVER (PARTITION BY p.route_ran, p.cov) AS sep_over_exp
FROM plays p
""")

per_route = con.execute(f"""
SELECT nfl_id, nfl_position, draft_year, route_ran,
       count(*)                      AS routes,
       count(sep)                    AS pass_fwd,
       avg(sep)                      AS mean_sep,
       avg(sep_over_exp)             AS sep_over_exp,
       avg(cushion)                  AS mean_cushion,
       sum(target)                   AS targets
FROM plays_x
GROUP BY ALL
HAVING count(sep) >= {MIN_ROUTE_PLAYS}
""").df()
per_route.to_parquet(HERE / "work" / "wr_route_outcomes.parquet")

per_player = con.execute(f"""
SELECT nfl_id, nfl_position, draft_year,
       count(*)                      AS routes,
       count(sep)                    AS pass_fwd,
       avg(sep)                      AS mean_sep,
       avg(sep_over_exp)             AS sep_over_exp,
       avg(cushion)                  AS mean_cushion,
       sum(target)::DOUBLE / count(*) AS target_rate,
       count(DISTINCT route_ran)     AS route_types
FROM plays_x
GROUP BY ALL
HAVING count(sep) >= {MIN_PLAYS}
""").df()
per_player.to_parquet(HERE / "work" / "wr_player_outcomes.parquet")

print("players with >=", MIN_PLAYS, "pass-forward routes:", len(per_player))
print(per_player.groupby("nfl_position").size().to_string())
print("player-route rows with >=", MIN_ROUTE_PLAYS, ":", len(per_route))
print(per_route.groupby("route_ran").size().sort_values(ascending=False).to_string())
print(per_player[["pass_fwd", "mean_sep", "sep_over_exp", "target_rate"]].describe().round(2).to_string())

_log.stop()

## 4. Receivers: do route-drill movement features relate to separation?

WR test: do combine route-drill features relate to NFL separation over expected?

Outcome: sep_over_exp (outcomes.py), WRs with >= 30 pass-forward routes.
Predictors: stopwatch/size numbers from combine_results (baseline) and sensor features
from the sharp-break route drills (features.py).

1. Spearman correlation of every predictor with the outcome, with a bootstrap 95% interval
   and a Benjamini-Hochberg adjusted p-value across all predictors tested.
2. Repeated 5-fold cross-validated ridge regression: baseline predictors alone versus
   baseline plus sensor features, on the same players.

In [ ]:
_log = _Tee('04_wr_overall')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(0)
SHARP = ("CURL_ROUTE_RIGHT", "COMEBACK_ROUTE_RIGHT", "SLOT_STRIKE_ROUTE_LEFT")
BASELINE = ["forty", "ten_yd_split", "three_cone", "short_shuttle", "vertical", "broad_jump", "weight"]

con = duckdb.connect()
out = con.execute(
    "SELECT nfl_id, pass_fwd, sep_over_exp FROM 'work/wr_player_outcomes.parquet' WHERE nfl_position = 'WR' ORDER BY nfl_id"
).df()
comb = con.execute(
    "SELECT nfl_id, forty, ten_yd_split, three_cone, short_shuttle, vertical, broad_jump, "
    "combine_weight AS weight FROM 'work/combine_results.parquet' ORDER BY nfl_id"
).df()
feat = pd.read_parquet(HERE / "work" / "route_features.parquet")
feat = feat[(feat.drill_type == "SKILL_DRILLS_WR") & feat.drill_name.isin(SHARP)]
sens = feat.groupby("nfl_id").agg(
    sharp_speed_lost=("speed_lost", "mean"),
    sharp_reaccel_s=("reaccel_s", "mean"),
    sharp_break_angle=("break_angle", "mean"),
    sharp_entry_speed=("entry_speed", "mean"),
    sharp_attempts=("event_id", "count"),
).reset_index()
peak = (pd.read_parquet(HERE / "work" / "route_features.parquet")
        .query("drill_type == 'SKILL_DRILLS_WR'").groupby("nfl_id").peak_speed.max()
        .rename("wr_peak_speed").reset_index())
df = out.merge(comb, on="nfl_id", how="left").merge(sens, on="nfl_id", how="left").merge(peak, on="nfl_id", how="left")
SENSOR = ["sharp_speed_lost", "sharp_reaccel_s", "sharp_break_angle", "sharp_entry_speed", "wr_peak_speed"]
print(f"WRs with outcome: {len(df)}")
print("non-missing per predictor:", df[BASELINE + SENSOR].notna().sum().to_dict())

# 1. correlations
rows = []
for c in BASELINE + SENSOR:
    d = df[[c, "sep_over_exp"]].dropna()
    r = spearmanr(d[c], d.sep_over_exp)
    boots = []
    for _ in range(2000):
        s = d.sample(len(d), replace=True, random_state=int(RNG.integers(1 << 31)))
        boots.append(spearmanr(s[c], s.sep_over_exp)[0])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    rows.append({"predictor": c, "n": len(d), "rho": r[0], "ci_lo": lo, "ci_hi": hi, "p": r[1]})
res = pd.DataFrame(rows)
m = len(res)
srt = res.p.sort_values()
adj = (srt * m / np.arange(1, m + 1))[::-1].cummin()[::-1].clip(upper=1)
res["p_bh"] = adj.reindex(res.index)
print("\nSpearman vs sep_over_exp (95% bootstrap CI):")
print(res.round(3).to_string(index=False))

# 2. cross-validated comparison on the same players
def cv_r2(cols: list[str], data: pd.DataFrame) -> tuple[float, float]:
    X, y = data[cols].to_numpy(dtype=float), data.sep_over_exp.to_numpy(dtype=float)
    scores = []
    for tr, te in RepeatedKFold(n_splits=5, n_repeats=20, random_state=0).split(X):
        med = np.nanmedian(X[tr], axis=0)
        Xtr, Xte = np.where(np.isnan(X[tr]), med, X[tr]), np.where(np.isnan(X[te]), med, X[te])
        mdl = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 3, 20))).fit(Xtr, y[tr])
        pred = mdl.predict(Xte)
        scores.append(1 - ((y[te] - pred) ** 2).sum() / ((y[te] - y[tr].mean()) ** 2).sum())
    return float(np.mean(scores)), float(np.std(scores))

need = ["forty", "vertical", "broad_jump", "weight"]   # 3-cone and shuttle exist for only 27 and 34 WRs
sub = df.dropna(subset=need + ["sharp_speed_lost", "sharp_reaccel_s"]).copy()
print(f"\nCross-validated R^2 (5-fold x 20) on {len(sub)} WRs with all of {need} and sensor features:")
for name, cols in [("baseline: " + "+".join(need), need),
                   ("baseline + sensor", need + ["sharp_speed_lost", "sharp_reaccel_s", "sharp_entry_speed"]),
                   ("sensor only", ["sharp_speed_lost", "sharp_reaccel_s", "sharp_entry_speed"])]:
    mu, sd = cv_r2(cols, sub)
    print(f"  {name:55s} R2 = {mu:+.3f}  (sd across folds {sd:.3f})")

_log.stop()

## 5. Receivers: each route paired with its closest drill, with controls

Route-matched WR test with controls.

Each NFL route is paired with the combine drill that most resembles it, and the sensor
features from that drill are tested against the player's separation over expected on that
route only. Weight, height and the share of routes run from the slot are partialled out of
both sides first (rank-transform, regress on the controls, correlate the residuals).

The pairings are my judgement of which drill looks like which route and were fixed before
looking at results: HITCH-curl/comeback, OUT-speed out, SLANT-slant, IN-dagger,
POST and CORNER-post-corner, GO-go (peak speed only).

In [ ]:
_log = _Tee('05_wr_route_matched')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, rankdata

RNG = np.random.default_rng(0)
PAIRS = {
    "HITCH": ["CURL_ROUTE_RIGHT", "COMEBACK_ROUTE_RIGHT"],
    "OUT": ["SPEED_OUT_ROUTE_LEFT", "SPEED_OUT_LEFT"],
    "SLANT": ["SLANT_ROUTE_LEFT"],
    "IN": ["DAGGER_ROUTE_LEFT"],
    "POST": ["POST_CORNER_ROUTE_LEFT", "POST_CORNER_ROUTE_RIGHT"],
    "CORNER": ["POST_CORNER_ROUTE_LEFT", "POST_CORNER_ROUTE_RIGHT"],
    "GO": ["GO_ROUTE_RIGHT"],
}
CUT_FEATURES = ["speed_lost", "reaccel_s", "break_angle"]
GO_FEATURES = ["peak_speed"]
CONTROLS = ["weight", "height", "slot_share"]

con = duckdb.connect()
route_out = con.execute(
    "SELECT nfl_id, route_ran, sep_over_exp, pass_fwd FROM 'work/wr_route_outcomes.parquet' WHERE nfl_position = 'WR' ORDER BY nfl_id, route_ran"
).df()
ctrl = con.execute("""
SELECT c.nfl_id, c.combine_weight AS weight, c.combine_height AS height,
       s.slot_share
FROM 'work/combine_results.parquet' c
JOIN (
  SELECT nfl_id, avg((lined_up_position = 'SLOT_WR')::INT) AS slot_share
  FROM 'work/player_play.parquet'
  WHERE route_ran IS NOT NULL AND play_nullified_by_penalty = 'N'
  GROUP BY nfl_id
) s USING (nfl_id)
ORDER BY c.nfl_id
""").df()
feat = pd.read_parquet(HERE / "work" / "route_features.parquet")
feat = feat[feat.drill_type == "SKILL_DRILLS_WR"]


def residualize(v: np.ndarray, Z: np.ndarray) -> np.ndarray:
    Z1 = np.column_stack([np.ones(len(Z)), Z])
    beta, *_ = np.linalg.lstsq(Z1, v, rcond=None)
    return v - Z1 @ beta


def partial_rank_corr(x: np.ndarray, y: np.ndarray, Z: np.ndarray) -> float:
    Zr = np.column_stack([rankdata(Z[:, i]) for i in range(Z.shape[1])])
    return pearsonr(residualize(rankdata(x), Zr), residualize(rankdata(y), Zr))[0]


rows = []
for route, drills in PAIRS.items():
    f = feat[feat.drill_name.isin(drills)].groupby("nfl_id")[CUT_FEATURES + GO_FEATURES].mean().reset_index()
    d = (route_out[route_out.route_ran == route].merge(f, on="nfl_id").merge(ctrl, on="nfl_id")
         .replace([np.inf, -np.inf], np.nan))
    for ft in (GO_FEATURES if route == "GO" else CUT_FEATURES):
        dd = d.dropna(subset=[ft, "sep_over_exp"] + CONTROLS)
        if len(dd) < 20:
            continue
        x, y, Z = dd[ft].to_numpy(float), dd.sep_over_exp.to_numpy(float), dd[CONTROLS].to_numpy(float)
        raw = pearsonr(rankdata(x), rankdata(y))[0]
        part = partial_rank_corr(x, y, Z)
        boots = []
        for _ in range(2000):
            i = RNG.integers(0, len(dd), len(dd))
            boots.append(partial_rank_corr(x[i], y[i], Z[i]))
        lo, hi = np.percentile(boots, [2.5, 97.5])
        p = 2 * (np.mean(np.array(boots) <= 0) if part > 0 else np.mean(np.array(boots) >= 0))
        rows.append({"route": route, "feature": ft, "n": len(dd), "raw_rho": raw,
                     "partial_rho": part, "ci_lo": lo, "ci_hi": hi, "p_boot": min(p, 1.0)})

res = pd.DataFrame(rows)
m = len(res)
srt = res.p_boot.sort_values()
adj = (srt * m / np.arange(1, m + 1))[::-1].cummin()[::-1].clip(upper=1)
res["p_bh"] = adj.reindex(res.index)
pd.set_option("display.width", 200)
print(f"{m} tests; controls partialled out: {CONTROLS}")
print(res.round(3).to_string(index=False))
res.to_csv(HERE / "work" / "wr_matched_results.csv", index=False)

_log.stop()

## 6. Linemen movement features

Drill-general movement features for OL and DL combine drills, from positions.

Unlike the route drills there is no single break to find, so the features describe how
the athlete gets going and how much they change direction:

  start         first frame where smoothed speed reaches START_SPEED
  speed_0p5s    speed 0.5 s after start
  burst_dist_1s distance covered in the first 1.0 s after start
  peak_accel_1s highest acceleration (rate of change of speed) in the first 1.0 s
  peak_speed    highest speed in the attempt
  mean_speed    mean speed while moving
  turn_rate     total absolute heading change per second while moving (direction changes)

In [ ]:
_log = _Tee('06_trench_features')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


START_SPEED = 1.0   # yd/s
MOVING = 1.0        # yd/s
GROUPS = {
    "OL": ("SKILL_DRILLS_OL", {
        "PASS_PRO_MIRROR": ("PASS_PRO_MIRROR_DRILL", "PASS_PRO-MIRROR_DRILL"),
        "PASS_RUSH_DROP": ("PASS_RUSH_DROP",),
        "WAVE_SLIDE": ("FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE",),
    }),
    "DL": ("SKILL_DRILLS_DL", {
        "PASS_RUSH": ("PASS_RUSH_DRILL",),
        "FRONT_START_WAVE": ("FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION",),
        "FOUR_BAG": ("FOUR_BAG_AGILITY_DRILL",),
        "BODY_CONTROL": ("BODY_CONTROL_DRILL",),
    }),
}


def attempt(g: pd.DataFrame) -> dict | None:
    if len(g) < 15:
        return None
    x, y = _smooth(g["x"].to_numpy()), _smooth(g["y"].to_numpy())
    vx, vy = np.gradient(x, DT), np.gradient(y, DT)
    speed = np.hypot(vx, vy)
    acc = np.gradient(speed, DT)
    go = np.nonzero(speed >= START_SPEED)[0]
    if len(go) == 0:
        return None
    s = int(go[0])
    if s + 10 >= len(speed):
        return None
    step = np.hypot(np.diff(x), np.diff(y))
    heading = np.unwrap(np.arctan2(vy, vx))
    mv = speed[1:] >= MOVING
    dh = np.abs(np.diff(heading))[mv[: len(heading) - 1]] if mv.any() else np.array([0.0])
    # Start-free features: the best window anywhere in the attempt, so they do not depend on
    # which frame is called the start. "_s" versions use the file's own speed and acceleration.
    sg, ag = g["s"].to_numpy(float), g["a"].to_numpy(float)
    xr, yr = g["x"].to_numpy(float), g["y"].to_numpy(float)
    n = len(g)
    net10 = np.hypot(xr[10:] - xr[:-10], yr[10:] - yr[:-10]) if n > 10 else np.array([0.0])
    gain5 = speed[5:] - speed[:-5]
    gain5_s = sg[5:] - sg[:-5]
    path10_s = np.convolve(sg * DT, np.ones(10), mode="valid") if n > 10 else np.array([0.0])
    top3 = np.sort(ag)[-3:].mean()
    return {
        "frames": len(g),
        "best_net_1s": float(net10.max()),
        "best_gain_0p5s": float(gain5.max()),
        "best_gain_0p5s_s": float(gain5_s.max()),
        "best_path_1s_s": float(path10_s.max()),
        "peak_speed_s": float(sg.max()),
        "peak_accel_s": float(ag.max()),
        "top3_accel_s": float(top3),
        "speed_0p5s": float(speed[s + 5]),
        "burst_dist_1s": float(step[s:s + 10].sum()),
        "peak_accel_1s": float(acc[s:s + 10].max()),
        "peak_speed": float(speed.max()),
        "mean_speed": float(speed[speed >= MOVING].mean()),
        "turn_rate": float(np.degrees(dh.sum()) / max(mv.sum() * DT, DT)),
    }


def build() -> pd.DataFrame:
    con = duckdb.connect()
    parts = []
    for side, (dtype, drills) in GROUPS.items():
        names = [n for v in drills.values() for n in v]
        df = con.execute(
            "SELECT event_id, nfl_id, drill_name, time, x, y, s, a FROM 'work/combine_tracking.parquet' "
            "WHERE entity_type = 'PLAYER' AND drill_type = ? AND drill_name IN (SELECT unnest(?)) "
            "ORDER BY event_id, nfl_id, time", [dtype, names]
        ).df()
        key = {n: k for k, v in drills.items() for n in v}
        for (ev, nid), g in df.groupby(["event_id", "nfl_id"], sort=False):
            f = attempt(g)
            if f:
                parts.append({"side": side, "drill": key[g.drill_name.iloc[0]], "event_id": ev, "nfl_id": nid, **f})
    return pd.DataFrame(parts)


if __name__ == "__main__":
    out = build()
    out.to_parquet(HERE / "work" / "trench_features.parquet")
    print(len(out), "attempts")
    pd.set_option("display.width", 200)
    print(out.groupby(["side", "drill"]).agg(n=("nfl_id", "size"), players=("nfl_id", "nunique"),
          speed_0p5s=("speed_0p5s", "median"), burst=("burst_dist_1s", "median"),
          accel=("peak_accel_1s", "median"), peak=("peak_speed", "median"),
          turn=("turn_rate", "median")).round(2).to_string())

_log.stop()

## 7. Linemen outcomes

NFL trench outcomes from player_play, one row per player.

OL (pass blocking). Rows where pressure_allowed is recorded.
  ol_pressure_rate_over_exp   pressure_allowed minus the league-wide logistic prediction from
                              dropback_duration and pass_rushers_encountered (longer drop-backs
                              and more rushers allow more pressure whoever is blocking).
  ol_pressure_prob_over_exp   the same for peak_pressure_probability_allowed, a continuous
                              measure that is less noisy than a yes/no pressure.

DL (pass rush). Rows where the defender lined up as EDGE or INTERIOR_LINE with a recorded
get-off or pass-rush result.
  dl_get_off_over_pos         player_get_off (a time in seconds, 0 to 2; pressured plays
                              average 0.88 against 1.02 for the rest, so lower is probably
                              faster) minus the average for the same lined-up position.
  dl_pressure_rate_over_pos   whether the rusher got pressure (time_to_pressure recorded) per
                              pass-rush play, minus the average for the same position.

Plays voided by a penalty are excluded. The meaning of get-off and of the pass-rush
denominator are inferred from the data, not from a data dictionary.

In [ ]:
_log = _Tee('07_trench_outcomes')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression

MIN_OL = 100
MIN_DL_GETOFF = 50
MIN_DL_RUSH = 100

con = duckdb.connect()
ol = con.execute("""
SELECT nfl_id, lined_up_position AS pos, pressure_allowed::INT AS pressure,
       peak_pressure_probability_allowed AS ppp, dropback_duration AS drop, pass_rushers_encountered AS rushers
FROM 'work/player_play.parquet'
WHERE play_nullified_by_penalty = 'N' AND pressure_allowed IS NOT NULL
  AND dropback_duration IS NOT NULL AND pass_rushers_encountered IS NOT NULL
""").df()
X = ol[["drop", "rushers"]].to_numpy(float)
ol["pressure_exp"] = LogisticRegression(max_iter=1000).fit(X, ol.pressure).predict_proba(X)[:, 1]
has = ol.ppp.notna()
ol["ppp_exp"] = np.nan
ol.loc[has, "ppp_exp"] = LinearRegression().fit(X[has], ol.ppp[has]).predict(X[has])
ol["tackle"] = (ol.pos == "T").astype(int)
ol_p = (ol.groupby("nfl_id").agg(
            ol_plays=("pressure", "size"),
            ol_pressure_rate=("pressure", "mean"),
            ol_pressure_rate_over_exp=("pressure", lambda s: float((s - ol.loc[s.index, "pressure_exp"]).mean())),
            ol_pressure_prob_over_exp=("ppp", lambda s: float((s - ol.loc[s.index, "ppp_exp"]).mean())),
            tackle_share=("tackle", "mean"))
        .query("ol_plays >= @MIN_OL").reset_index())
ol_p.to_parquet(HERE / "work" / "ol_outcomes.parquet")

dl = con.execute("""
SELECT nfl_id, lined_up_position AS pos, player_get_off AS getoff,
       CASE WHEN sack IS NOT NULL THEN (time_to_pressure IS NOT NULL)::INT END AS pressure
FROM 'work/player_play.parquet'
WHERE play_nullified_by_penalty = 'N' AND lined_up_position IN ('EDGE', 'INTERIOR_LINE')
""").df()
dl["pressure"] = dl.pressure.astype(float)
dl["getoff"] = dl.getoff.astype(float)
dl["getoff_exp"] =dl.groupby("pos").getoff.transform("mean")
dl["pressure_exp"] = dl.groupby("pos").pressure.transform("mean")
dl["edge"] = (dl.pos == "EDGE").astype(int)
dl_p = (dl.groupby("nfl_id").agg(
            dl_getoff_plays=("getoff", "count"),
            dl_rush_plays=("pressure", "count"),
            dl_get_off=("getoff", "mean"),
            dl_get_off_over_pos=("getoff", lambda s: float((s - dl.loc[s.index, "getoff_exp"]).mean())),
            dl_pressure_rate=("pressure", "mean"),
            dl_pressure_rate_over_pos=("pressure", lambda s: float((s - dl.loc[s.index, "pressure_exp"]).mean())),
            edge_share=("edge", "mean"))
        .reset_index())
dl_p.to_parquet(HERE / "work" / "dl_outcomes.parquet")

print(f"OL players with >= {MIN_OL} pass-block plays: {len(ol_p)}")
print(ol_p[["ol_plays", "ol_pressure_rate", "ol_pressure_rate_over_exp", "ol_pressure_prob_over_exp"]].describe().round(3).to_string())
print(f"\nDL players: get-off >= {MIN_DL_GETOFF}: {(dl_p.dl_getoff_plays >= MIN_DL_GETOFF).sum()}, "
      f"rush plays >= {MIN_DL_RUSH}: {(dl_p.dl_rush_plays >= MIN_DL_RUSH).sum()}")
print(dl_p[["dl_getoff_plays", "dl_rush_plays", "dl_get_off", "dl_get_off_over_pos", "dl_pressure_rate",
            "dl_pressure_rate_over_pos"]].describe().round(3).to_string())

_log.stop()

## 8. Statistics helpers

Partial rank correlation with a bootstrap interval, and the Benjamini-Hochberg adjustment used by every family of tests.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, rankdata


def _residualize(v: np.ndarray, Z: np.ndarray) -> np.ndarray:
    Z1 = np.column_stack([np.ones(len(Z)), Z])
    beta, *_ = np.linalg.lstsq(Z1, v, rcond=None)
    return v - Z1 @ beta


def partial_rank_corr(x: np.ndarray, y: np.ndarray, Z: np.ndarray | None) -> float:
    """Correlation of the ranks of x and y after removing the ranks of the controls Z."""
    rx, ry = rankdata(x), rankdata(y)
    if Z is not None and Z.shape[1]:
        Zr = np.column_stack([rankdata(Z[:, i]) for i in range(Z.shape[1])])
        rx, ry = _residualize(rx, Zr), _residualize(ry, Zr)
    if np.ptp(rx) == 0 or np.ptp(ry) == 0:
        return np.nan
    return pearsonr(rx, ry)[0]


def bootstrap_partial(x, y, Z, rng, n_boot: int = 2000) -> dict:
    est = partial_rank_corr(x, y, Z)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        boots.append(partial_rank_corr(x[i], y[i], None if Z is None else Z[i]))
    b = np.array(boots)
    b = b[~np.isnan(b)]
    lo, hi = np.percentile(b, [2.5, 97.5])
    p = 2 * min(np.mean(b <= 0), np.mean(b >= 0))
    return {"rho": est, "ci_lo": lo, "ci_hi": hi, "p_boot": min(max(p, 1.0 / len(b)), 1.0), "boot_ok": len(b)}


def bh_adjust(p: pd.Series) -> pd.Series:
    m = len(p)
    srt = p.sort_values()
    adj = (srt * m / np.arange(1, m + 1))[::-1].cummin()[::-1].clip(upper=1)
    return adj.reindex(p.index)

## 9. Linemen, first pass: five start-based features

Trench test: do OL/DL combine drill movement features relate to NFL pass-blocking and
pass-rush results?

Sensor features (trench_features.py) are z-scored within each drill and averaged over the
player's matched drills. Each is tested against two outcomes per side (trench_outcomes.py)
with weight, height and the share of snaps at the main position (tackle for OL, edge for DL)
partialled out. The 20 sensor tests are one family for the Benjamini-Hochberg adjustment.
The stopwatch baselines (10-yard split, 40) are reported beside them, uncontrolled and
controlled, and are not part of the family.

In [ ]:
_log = _Tee('08_trench_first_pass')
sys.argv = ['notebook', '--original']
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


RNG = np.random.default_rng(0)
# Second pass: three start-free features chosen from the reliability table (reliability.py).
# `python test_trench.py --original` runs the first pass with the five start-based features.
FEATURES = ["best_net_1s", "best_gain_0p5s", "peak_speed_s"]
if "--original" in sys.argv:
    FEATURES = ["speed_0p5s", "burst_dist_1s", "peak_accel_1s", "peak_speed", "turn_rate"]
MIN_OL, MIN_DL_GETOFF, MIN_DL_RUSH = 100, 50, 100

con = duckdb.connect()
comb = con.execute(
    "SELECT nfl_id, combine_position AS cpos, combine_weight AS weight, combine_height AS height, "
    "ten_yd_split, forty FROM 'work/combine_results.parquet'"
).df()
tf = pd.read_parquet(HERE / "work" / "trench_features.parquet")
tf[FEATURES] = tf.groupby(["side", "drill"])[FEATURES].transform(lambda s: (s - s.mean()) / s.std())
pooled = tf.groupby(["side", "nfl_id"])[FEATURES].mean().reset_index()

ol_o = pd.read_parquet(HERE / "work" / "ol_outcomes.parquet")
dl_o = pd.read_parquet(HERE / "work" / "dl_outcomes.parquet")
SIDES = {
    "OL": dict(out=ol_o.query("ol_plays >= @MIN_OL"), cpos="OL", share="tackle_share",
               outcomes=["ol_pressure_rate_over_exp", "ol_pressure_prob_over_exp"]),
    "DL": dict(out=dl_o, cpos="DL", share="edge_share",
               outcomes=["dl_get_off_over_pos", "dl_pressure_rate_over_pos"]),
}
MINS = {"dl_get_off_over_pos": ("dl_getoff_plays", MIN_DL_GETOFF),
        "dl_pressure_rate_over_pos": ("dl_rush_plays", MIN_DL_RUSH)}

rows, base = [], []
for side, cfg in SIDES.items():
    d0 = (cfg["out"].merge(comb[comb.cpos == cfg["cpos"]], on="nfl_id")
          .merge(pooled[pooled.side == side].drop(columns="side"), on="nfl_id", how="left"))
    ctrl_cols = ["weight", "height", cfg["share"]]
    for oc in cfg["outcomes"]:
        d1 = d0
        if oc in MINS:
            col, mn = MINS[oc]
            d1 = d0[d0[col] >= mn]
        for ft in FEATURES + ["ten_yd_split", "forty"]:
            dd = d1.dropna(subset=[ft, oc] + ctrl_cols)
            if len(dd) < 25:
                continue
            x, y, Z = dd[ft].to_numpy(float), dd[oc].to_numpy(float), dd[ctrl_cols].to_numpy(float)
            r = bootstrap_partial(x, y, Z, RNG)
            r_raw = bootstrap_partial(x, y, None, RNG) if ft in ("ten_yd_split", "forty") else {"rho": np.nan}
            rec = {"side": side, "outcome": oc, "feature": ft, "n": len(dd), "raw_rho": r_raw["rho"],
                   "partial_rho": r["rho"], "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"], "p_boot": r["p_boot"]}
            (base if ft in ("ten_yd_split", "forty") else rows).append(rec)

res = pd.DataFrame(rows)
res["p_bh"] = bh_adjust(res.p_boot)
pd.set_option("display.width", 220)
print(f"Sensor features: {len(res)} tests (one family)")
print(res.drop(columns="raw_rho").round(3).to_string(index=False))
print("\nStopwatch baselines (not in the family):")
print(pd.DataFrame(base).round(3).to_string(index=False))
res.to_csv(HERE / "work" / ("trench_results_original.csv" if "--original" in sys.argv else "trench_results.csv"), index=False)

_log.stop()

## 10. Overlap with the stopwatch and repeat reliability, first-pass features

How much of each sensor feature is already explained by the stopwatch and size numbers?

For every sensor feature (WR sharp-cut drills, OL drills, DL drills) this fits a
cross-validated ridge regression of the feature on forty, 10-yard split, vertical, broad
jump, weight and height, and reports the out-of-sample R^2 (5-fold, 20 repeats). It also
reports the Spearman correlation with the 40 alone. A feature with a low R^2 carries
information the stopwatch does not; a high R^2 means it largely repeats it.

In [ ]:
_log = _Tee('09_overlap_first_pass')
sys.argv = ['notebook', '--original']
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

STOPWATCH = ["forty", "ten_yd_split", "vertical", "broad_jump", "combine_weight", "combine_height"]
SHARP = ("CURL_ROUTE_RIGHT", "COMEBACK_ROUTE_RIGHT", "SLOT_STRIKE_ROUTE_LEFT")
TRENCH = ["best_net_1s", "best_gain_0p5s", "peak_speed_s"]
if "--original" in sys.argv:   # the first-pass start-based features
    TRENCH = ["speed_0p5s", "burst_dist_1s", "peak_accel_1s", "peak_speed", "turn_rate"]

con = duckdb.connect()
comb = con.execute(
    "SELECT nfl_id, combine_position AS cpos, forty, ten_yd_split, vertical, broad_jump, "
    "combine_weight, combine_height FROM 'work/combine_results.parquet'"
).df()

wr = pd.read_parquet(HERE / "work" / "route_features.parquet")
wr = wr[(wr.drill_type == "SKILL_DRILLS_WR") & wr.drill_name.isin(SHARP)]
wr = wr.groupby("nfl_id")[["speed_lost", "reaccel_s", "break_angle", "entry_speed", "peak_speed"]].mean().reset_index()

tf = pd.read_parquet(HERE / "work" / "trench_features.parquet")
tf[TRENCH] = tf.groupby(["side", "drill"])[TRENCH].transform(lambda s: (s - s.mean()) / s.std())
tr = {s: g.groupby("nfl_id")[TRENCH].mean().reset_index() for s, g in tf.groupby("side")}

GROUPS = [("WR sharp cuts", "WR", wr, list(wr.columns[1:])),
          ("OL drills", "OL", tr["OL"], TRENCH),
          ("DL drills", "DL", tr["DL"], TRENCH)]


def cv_r2(X: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    scores = []
    for a, b in RepeatedKFold(n_splits=5, n_repeats=20, random_state=0).split(X):
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 3, 20))).fit(X[a], y[a])
        scores.append(1 - ((y[b] - m.predict(X[b])) ** 2).sum() / ((y[b] - y[a].mean()) ** 2).sum())
    return float(np.mean(scores)), float(np.std(scores))


rows = []
for label, cpos, feats, cols in GROUPS:
    d = comb[comb.cpos == cpos].merge(feats, on="nfl_id").dropna(subset=STOPWATCH)
    for c in cols:
        dd = d.dropna(subset=[c])
        X, y = dd[STOPWATCH].to_numpy(float), dd[c].to_numpy(float)
        r2, sd = cv_r2(X, y)
        rho = spearmanr(dd.forty, dd[c])[0]
        rows.append({"group": label, "feature": c, "n": len(dd), "cv_r2_stopwatch": r2,
                     "fold_sd": sd, "rho_with_forty": rho})

# Repeat-attempt reliability: correlation between an athlete's first and second attempt
# of the same drill. A feature that does not repeat cannot be explained by anything.
rel = []
for (side, drill), g in tf.groupby(["side", "drill"]):
    g = g.sort_values("event_id")
    g["k"] = g.groupby("nfl_id").cumcount()
    a, b = g[g.k == 0].set_index("nfl_id"), g[g.k == 1].set_index("nfl_id")
    both = a.index.intersection(b.index)
    for c in TRENCH:
        if len(both) >= 20:
            rel.append({"side": side, "drill": drill, "feature": c, "athletes_with_2": len(both),
                        "attempt_corr": spearmanr(a.loc[both, c], b.loc[both, c])[0]})
reliability = pd.DataFrame(rel)

res = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(f"Stopwatch + size predictors: {STOPWATCH}")
print(res.round(3).to_string(index=False))
res.to_csv(HERE / "work" / "overlap_results.csv", index=False)
print("\nRepeat-attempt reliability (Spearman, first vs second attempt, same athlete and drill):")
print(reliability.round(3).to_string(index=False))

_log.stop()

## 11. Repeat reliability of every linemen feature

Repeat-attempt reliability of every trench feature, per drill.

For athletes with two attempts at the same drill, the Spearman correlation between the first
and second attempt. Also the Spearman-Brown value for the mean of two attempts
(2r / (1 + r)), which is the reliability of a feature averaged over both.

In [ ]:
_log = _Tee('10_reliability')
sys.argv = ['notebook']
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

tf = pd.read_parquet(HERE / "work" / "trench_features.parquet")
FEATS = [c for c in tf.columns if c not in ("side", "drill", "event_id", "nfl_id", "frames")]

rows = []
for (side, drill), g in tf.groupby(["side", "drill"]):
    g = g.sort_values("event_id")
    g["k"] = g.groupby("nfl_id").cumcount()
    a, b = g[g.k == 0].set_index("nfl_id"), g[g.k == 1].set_index("nfl_id")
    both = a.index.intersection(b.index)
    if len(both) < 20:
        continue
    for c in FEATS:
        r = spearmanr(a.loc[both, c], b.loc[both, c])[0]
        rows.append({"side": side, "drill": drill, "feature": c, "n": len(both),
                     "attempt_corr": r, "mean_of_two": 2 * r / (1 + r) if r > -1 else float("nan")})
res = pd.DataFrame(rows).sort_values(["side", "attempt_corr"], ascending=[True, False])
pd.set_option("display.width", 200)
print(res.round(3).to_string(index=False))
res.to_csv(HERE / "work" / "reliability.csv", index=False)

_log.stop()

## 12. Linemen, second pass: three start-free features

Trench test: do OL/DL combine drill movement features relate to NFL pass-blocking and
pass-rush results?

Sensor features (trench_features.py) are z-scored within each drill and averaged over the
player's matched drills. Each is tested against two outcomes per side (trench_outcomes.py)
with weight, height and the share of snaps at the main position (tackle for OL, edge for DL)
partialled out. The 20 sensor tests are one family for the Benjamini-Hochberg adjustment.
The stopwatch baselines (10-yard split, 40) are reported beside them, uncontrolled and
controlled, and are not part of the family.

In [ ]:
_log = _Tee('11_trench_second_pass')
sys.argv = ['notebook']
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


RNG = np.random.default_rng(0)
# Second pass: three start-free features chosen from the reliability table (reliability.py).
# `python test_trench.py --original` runs the first pass with the five start-based features.
FEATURES = ["best_net_1s", "best_gain_0p5s", "peak_speed_s"]
if "--original" in sys.argv:
    FEATURES = ["speed_0p5s", "burst_dist_1s", "peak_accel_1s", "peak_speed", "turn_rate"]
MIN_OL, MIN_DL_GETOFF, MIN_DL_RUSH = 100, 50, 100

con = duckdb.connect()
comb = con.execute(
    "SELECT nfl_id, combine_position AS cpos, combine_weight AS weight, combine_height AS height, "
    "ten_yd_split, forty FROM 'work/combine_results.parquet'"
).df()
tf = pd.read_parquet(HERE / "work" / "trench_features.parquet")
tf[FEATURES] = tf.groupby(["side", "drill"])[FEATURES].transform(lambda s: (s - s.mean()) / s.std())
pooled = tf.groupby(["side", "nfl_id"])[FEATURES].mean().reset_index()

ol_o = pd.read_parquet(HERE / "work" / "ol_outcomes.parquet")
dl_o = pd.read_parquet(HERE / "work" / "dl_outcomes.parquet")
SIDES = {
    "OL": dict(out=ol_o.query("ol_plays >= @MIN_OL"), cpos="OL", share="tackle_share",
               outcomes=["ol_pressure_rate_over_exp", "ol_pressure_prob_over_exp"]),
    "DL": dict(out=dl_o, cpos="DL", share="edge_share",
               outcomes=["dl_get_off_over_pos", "dl_pressure_rate_over_pos"]),
}
MINS = {"dl_get_off_over_pos": ("dl_getoff_plays", MIN_DL_GETOFF),
        "dl_pressure_rate_over_pos": ("dl_rush_plays", MIN_DL_RUSH)}

rows, base = [], []
for side, cfg in SIDES.items():
    d0 = (cfg["out"].merge(comb[comb.cpos == cfg["cpos"]], on="nfl_id")
          .merge(pooled[pooled.side == side].drop(columns="side"), on="nfl_id", how="left"))
    ctrl_cols = ["weight", "height", cfg["share"]]
    for oc in cfg["outcomes"]:
        d1 = d0
        if oc in MINS:
            col, mn = MINS[oc]
            d1 = d0[d0[col] >= mn]
        for ft in FEATURES + ["ten_yd_split", "forty"]:
            dd = d1.dropna(subset=[ft, oc] + ctrl_cols)
            if len(dd) < 25:
                continue
            x, y, Z = dd[ft].to_numpy(float), dd[oc].to_numpy(float), dd[ctrl_cols].to_numpy(float)
            r = bootstrap_partial(x, y, Z, RNG)
            r_raw = bootstrap_partial(x, y, None, RNG) if ft in ("ten_yd_split", "forty") else {"rho": np.nan}
            rec = {"side": side, "outcome": oc, "feature": ft, "n": len(dd), "raw_rho": r_raw["rho"],
                   "partial_rho": r["rho"], "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"], "p_boot": r["p_boot"]}
            (base if ft in ("ten_yd_split", "forty") else rows).append(rec)

res = pd.DataFrame(rows)
res["p_bh"] = bh_adjust(res.p_boot)
pd.set_option("display.width", 220)
print(f"Sensor features: {len(res)} tests (one family)")
print(res.drop(columns="raw_rho").round(3).to_string(index=False))
print("\nStopwatch baselines (not in the family):")
print(pd.DataFrame(base).round(3).to_string(index=False))
res.to_csv(HERE / "work" / ("trench_results_original.csv" if "--original" in sys.argv else "trench_results.csv"), index=False)

_log.stop()

## 13. Overlap with the stopwatch, second-pass features

How much of each sensor feature is already explained by the stopwatch and size numbers?

For every sensor feature (WR sharp-cut drills, OL drills, DL drills) this fits a
cross-validated ridge regression of the feature on forty, 10-yard split, vertical, broad
jump, weight and height, and reports the out-of-sample R^2 (5-fold, 20 repeats). It also
reports the Spearman correlation with the 40 alone. A feature with a low R^2 carries
information the stopwatch does not; a high R^2 means it largely repeats it.

In [ ]:
_log = _Tee('12_overlap_second_pass')
sys.argv = ['notebook']
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

STOPWATCH = ["forty", "ten_yd_split", "vertical", "broad_jump", "combine_weight", "combine_height"]
SHARP = ("CURL_ROUTE_RIGHT", "COMEBACK_ROUTE_RIGHT", "SLOT_STRIKE_ROUTE_LEFT")
TRENCH = ["best_net_1s", "best_gain_0p5s", "peak_speed_s"]
if "--original" in sys.argv:   # the first-pass start-based features
    TRENCH = ["speed_0p5s", "burst_dist_1s", "peak_accel_1s", "peak_speed", "turn_rate"]

con = duckdb.connect()
comb = con.execute(
    "SELECT nfl_id, combine_position AS cpos, forty, ten_yd_split, vertical, broad_jump, "
    "combine_weight, combine_height FROM 'work/combine_results.parquet'"
).df()

wr = pd.read_parquet(HERE / "work" / "route_features.parquet")
wr = wr[(wr.drill_type == "SKILL_DRILLS_WR") & wr.drill_name.isin(SHARP)]
wr = wr.groupby("nfl_id")[["speed_lost", "reaccel_s", "break_angle", "entry_speed", "peak_speed"]].mean().reset_index()

tf = pd.read_parquet(HERE / "work" / "trench_features.parquet")
tf[TRENCH] = tf.groupby(["side", "drill"])[TRENCH].transform(lambda s: (s - s.mean()) / s.std())
tr = {s: g.groupby("nfl_id")[TRENCH].mean().reset_index() for s, g in tf.groupby("side")}

GROUPS = [("WR sharp cuts", "WR", wr, list(wr.columns[1:])),
          ("OL drills", "OL", tr["OL"], TRENCH),
          ("DL drills", "DL", tr["DL"], TRENCH)]


def cv_r2(X: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    scores = []
    for a, b in RepeatedKFold(n_splits=5, n_repeats=20, random_state=0).split(X):
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 3, 20))).fit(X[a], y[a])
        scores.append(1 - ((y[b] - m.predict(X[b])) ** 2).sum() / ((y[b] - y[a].mean()) ** 2).sum())
    return float(np.mean(scores)), float(np.std(scores))


rows = []
for label, cpos, feats, cols in GROUPS:
    d = comb[comb.cpos == cpos].merge(feats, on="nfl_id").dropna(subset=STOPWATCH)
    for c in cols:
        dd = d.dropna(subset=[c])
        X, y = dd[STOPWATCH].to_numpy(float), dd[c].to_numpy(float)
        r2, sd = cv_r2(X, y)
        rho = spearmanr(dd.forty, dd[c])[0]
        rows.append({"group": label, "feature": c, "n": len(dd), "cv_r2_stopwatch": r2,
                     "fold_sd": sd, "rho_with_forty": rho})

# Repeat-attempt reliability: correlation between an athlete's first and second attempt
# of the same drill. A feature that does not repeat cannot be explained by anything.
rel = []
for (side, drill), g in tf.groupby(["side", "drill"]):
    g = g.sort_values("event_id")
    g["k"] = g.groupby("nfl_id").cumcount()
    a, b = g[g.k == 0].set_index("nfl_id"), g[g.k == 1].set_index("nfl_id")
    both = a.index.intersection(b.index)
    for c in TRENCH:
        if len(both) >= 20:
            rel.append({"side": side, "drill": drill, "feature": c, "athletes_with_2": len(both),
                        "attempt_corr": spearmanr(a.loc[both, c], b.loc[both, c])[0]})
reliability = pd.DataFrame(rel)

res = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(f"Stopwatch + size predictors: {STOPWATCH}")
print(res.round(3).to_string(index=False))
res.to_csv(HERE / "work" / "overlap_results.csv", index=False)
print("\nRepeat-attempt reliability (Spearman, first vs second attempt, same athlete and drill):")
print(reliability.round(3).to_string(index=False))

_log.stop()

## 14. Replication across draft classes

Replication check across draft classes for the two leads from test_trench.py.

Fixed in advance, and only these two pairs are tested:
  DL: best_gain_0p5s  vs dl_pressure_rate_over_pos   (>= 100 pass-rush plays)
  OL: best_net_1s     vs ol_pressure_rate_over_exp   (>= 100 pass-block plays)
Same controls as before (weight, height, share of snaps at the main position). For each draft
class the partial rank correlation and a bootstrap interval; a lead counts as replicating
only if the sign matches in all three classes. Exploratory either way.

In [ ]:
_log = _Tee('13_replication')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


RNG = np.random.default_rng(0)
FEATS = ["best_net_1s", "best_gain_0p5s"]

con = duckdb.connect()
comb = con.execute(
    "SELECT nfl_id, draft_year, combine_position AS cpos, combine_weight AS weight, combine_height AS height "
    "FROM 'work/combine_results.parquet'"
).df()
tf = pd.read_parquet(HERE / "work" / "trench_features.parquet")
tf[FEATS] = tf.groupby(["side", "drill"])[FEATS].transform(lambda s: (s - s.mean()) / s.std())
pooled = tf.groupby(["side", "nfl_id"])[FEATS].mean().reset_index()
ol = pd.read_parquet(HERE / "work" / "ol_outcomes.parquet").query("ol_plays >= 100")
dl = pd.read_parquet(HERE / "work" / "dl_outcomes.parquet").query("dl_rush_plays >= 100")

LEADS = [("DL", "best_gain_0p5s", "dl_pressure_rate_over_pos", dl, "edge_share"),
         ("OL", "best_net_1s", "ol_pressure_rate_over_exp", ol, "tackle_share")]
rows = []
for side, ft, oc, out, share in LEADS:
    d = (out.merge(comb[comb.cpos == side], on="nfl_id")
         .merge(pooled[pooled.side == side].drop(columns="side"), on="nfl_id")
         .dropna(subset=[ft, oc, "weight", "height", share]))
    ctrl = ["weight", "height", share]
    groups = [("all", d)] + [(str(y), g) for y, g in d.groupby("draft_year")]
    for label, g in groups:
        if len(g) < 12:
            continue
        r = bootstrap_partial(g[ft].to_numpy(float), g[oc].to_numpy(float), g[ctrl].to_numpy(float), RNG)
        rows.append({"lead": f"{side} {ft} -> {oc}", "class": label, "n": len(g),
                     "partial_rho": r["rho"], "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"]})
res = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(res.round(3).to_string(index=False))
for lead, g in res.groupby("lead", sort=False):
    cls = g[g["class"] != "all"]
    same = (np.sign(cls.partial_rho) == np.sign(g[g["class"] == "all"].partial_rho.iloc[0])).all()
    print(f"{lead}: sign matches pooled in all {len(cls)} classes: {same}")

_log.stop()

## 15. Robustness checks on the offensive-line lead

Robustness checks on the OL lead: best_net_1s (best distance in any one second of the OL
drills) against pass-blocking pressure allowed, with weight, height and tackle share
partialled out.

Checks, all listed before running:
  1. Other outcome definitions: logistic-adjusted pressure rate (the original), raw pressure
     rate, sack-allowed rate, mean time to pressure on plays where pressure was allowed.
  2. Team effects: drop each team in turn; and subtract each team's mean outcome.
  3. Feature variants: the lead in each OL drill separately; a second distance measure.
  4. Play-count cutoffs: 50, 100, 200, 300 pass-block plays.
  5. Permutation: Freedman-Lane (permute the outcome after removing the controls), giving the
     chance of a correlation this large for this one test, and for the largest of the six OL
     tests (3 features x 2 outcomes) taken together.

In [ ]:
_log = _Tee('14_robustness_ol')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy.stats import rankdata


RNG = np.random.default_rng(1)
CTRL = ["weight", "height", "tackle_share"]
FEATS = ["best_net_1s", "best_gain_0p5s", "peak_speed_s"]

con = duckdb.connect()
plays = con.execute("""
SELECT nfl_id, team_abbr, lined_up_position AS pos, pressure_allowed::INT AS pressure,
       sack_allowed::INT AS sack, time_to_pressure_allowed AS ttp, peak_pressure_probability_allowed AS ppp
FROM 'work/player_play.parquet'
WHERE play_nullified_by_penalty = 'N' AND pressure_allowed IS NOT NULL
""").df()
print("OL plays:", len(plays), "| sack_allowed recorded:", int(plays.sack.notna().sum()),
      "| time_to_pressure_allowed recorded:", int(plays.ttp.notna().sum()))

team = (plays.groupby(["nfl_id", "team_abbr"]).size().reset_index(name="n")
        .sort_values("n", ascending=False).drop_duplicates("nfl_id")[["nfl_id", "team_abbr"]])
per = plays.groupby("nfl_id").agg(plays=("pressure", "size"), raw_pressure=("pressure", "mean"),
                                  sack_rate=("sack", "mean"), mean_ttp=("ttp", "mean"),
                                  n_ttp=("ttp", "count")).reset_index()
adj = pd.read_parquet(HERE / "work" / "ol_outcomes.parquet")[["nfl_id", "ol_pressure_rate_over_exp",
                                                              "ol_pressure_prob_over_exp", "tackle_share"]]
comb = con.execute("SELECT nfl_id, combine_position AS cpos, combine_weight AS weight, combine_height AS height "
                   "FROM 'work/combine_results.parquet'").df()
tf = pd.read_parquet(HERE / "work" / "trench_features.parquet")
tf = tf[tf.side == "OL"].copy()
ALLF = FEATS + ["best_path_1s_s"]
tf[ALLF] = tf.groupby("drill")[ALLF].transform(lambda s: (s - s.mean()) / s.std())
pooled = tf.groupby("nfl_id")[ALLF].mean().reset_index()

base = (per.merge(adj, on="nfl_id", how="inner").merge(team, on="nfl_id").merge(comb[comb.cpos == "OL"], on="nfl_id")
        .merge(pooled, on="nfl_id", how="left"))
print("OL players with outcome, combine and drill features:", int(base.dropna(subset=["best_net_1s"]).shape[0]))


def rho(d: pd.DataFrame, ft: str, oc: str, boot: bool = False):
    dd = d.dropna(subset=[ft, oc] + CTRL)
    if len(dd) < 20:
        return None
    x, y, Z = dd[ft].to_numpy(float), dd[oc].to_numpy(float), dd[CTRL].to_numpy(float)
    if boot:
        r = bootstrap_partial(x, y, Z, RNG, n_boot=1000)
        return len(dd), r["rho"], r["ci_lo"], r["ci_hi"]
    return len(dd), partial_rank_corr(x, y, Z), np.nan, np.nan


MAIN = base[base.plays >= 100]
LEAD = ("best_net_1s", "ol_pressure_rate_over_exp")
print("\n0. Original lead:", [round(v, 3) for v in rho(MAIN, *LEAD, boot=True)])

print("\n1. Other outcome definitions (best_net_1s):")
for oc in ["ol_pressure_rate_over_exp", "ol_pressure_prob_over_exp", "raw_pressure", "sack_rate"]:
    print(f"   {oc:28s} n,rho,ci:", [round(v, 3) for v in rho(MAIN, "best_net_1s", oc, boot=True)])
t = MAIN[MAIN.n_ttp >= 20]
r = rho(t, "best_net_1s", "mean_ttp", boot=True)
print(f"   mean_ttp (>=20 pressured plays) n,rho,ci:", [round(v, 3) for v in r] if r else "too few players")

print("\n2. Team effects:")
teams = MAIN.team_abbr.value_counts()
res = []
for tm in teams.index:
    r = rho(MAIN[MAIN.team_abbr != tm], *LEAD)
    res.append((tm, r[0], r[1]))
rs = np.array([x[2] for x in res])
print(f"   leave-one-team-out over {len(res)} teams: rho min {rs.min():.3f}, max {rs.max():.3f}, "
      f"share still negative {np.mean(rs < 0):.2f}")
worst = sorted(res, key=lambda z: z[2])[-3:]
print("   weakest when dropped:", [(a, b, round(c, 3)) for a, b, c in worst])
m2 = MAIN[MAIN.team_abbr.map(teams) >= 2].copy()
m2["oc_demeaned"] = m2[LEAD[1]] - m2.groupby("team_abbr")[LEAD[1]].transform("mean")
print("   team-mean removed (teams with >=2 players) n,rho,ci:",
      [round(v, 3) for v in rho(m2, LEAD[0], "oc_demeaned", boot=True)])

print("\n3. Feature variants:")
for dname, g in tf.groupby("drill"):
    g = g.groupby("nfl_id")[["best_net_1s"]].mean().reset_index().rename(columns={"best_net_1s": "f"})
    d = MAIN.drop(columns=["best_net_1s"]).merge(g, on="nfl_id")
    print(f"   best_net_1s in {dname:16s} n,rho,ci:", [round(v, 3) for v in rho(d, "f", LEAD[1], boot=True)])
print("   best_path_1s_s (file's own speed) n,rho,ci:", [round(v, 3) for v in rho(MAIN, "best_path_1s_s", LEAD[1], boot=True)])

print("\n4. Play-count cutoffs:")
for mn in (50, 100, 200, 300):
    r = rho(base[base.plays >= mn], *LEAD, boot=True)
    print(f"   >= {mn:3d} plays  n,rho,ci:", [round(v, 3) for v in r])

print("\n5. Permutation (Freedman-Lane, 5000 shuffles):")
d = MAIN.dropna(subset=FEATS + CTRL + ["ol_pressure_rate_over_exp", "ol_pressure_prob_over_exp"]).reset_index(drop=True)
Zr = np.column_stack([rankdata(d[c]) for c in CTRL])
Z1 = np.column_stack([np.ones(len(d)), Zr])
X = np.column_stack([_residualize(rankdata(d[f]), Zr) for f in FEATS])
Y = np.column_stack([_residualize(rankdata(d[o]), Zr) for o in ["ol_pressure_rate_over_exp", "ol_pressure_prob_over_exp"]])
Xn, Yn = X / np.linalg.norm(X, axis=0), Y / np.linalg.norm(Y, axis=0)
obs = (Xn.T @ Yn)
lead_obs = obs[0, 0]
mx_obs = np.abs(obs).max()
cnt_single = cnt_max = 0
N = 5000
for _ in range(N):
    p = RNG.permutation(len(d))
    c = Xn.T @ Yn[p]
    cnt_single += abs(c[0, 0]) >= abs(lead_obs)
    cnt_max += np.abs(c).max() >= mx_obs
print(f"   n={len(d)}; lead rho {lead_obs:.3f}; chance of |rho| this large for this one test: {(cnt_single + 1) / (N + 1):.3f}")
print(f"   largest |rho| over the 6 OL tests {mx_obs:.3f}; chance the largest of 6 is this big: {(cnt_max + 1) / (N + 1):.3f}")

_log.stop()

## 16. Defensive backs from game tracking

DB coverage pairs from game tracking, and a separation-allowed outcome.

The game tracking holds only the 497 draft-class players (usually 1 to 3 per play, no
opponents, no ball). So a DB's separation allowed can be measured only on plays where the
receiver he faces is also a draft-class player. A pair is kept when:
  - the DB lined up at CB or SLOT_CB and was in man coverage at the snap (coverage_assignment_at_snap),
  - the receiver ran a route for the other team and a pass was thrown (pass_forward event),
  - both are tracked, and the DB started within SNAP_MAX yards of the receiver.
The distance at the snap is known before the play develops, so choosing on it does not
select on how the play went. Whether this DB was really assigned to this receiver is not
known. As a check (not used to choose pairs), the share of pairs where the DB-receiver
distance at pass_forward equals the file's own separation_at_pass_forward (nearest defender)
is reported.

Outcome per pair: distance DB to receiver at pass_forward, in yards. Expected value: the
mean for the same route type, so the outcome is the pair distance minus that mean.

In [ ]:
_log = _Tee('15_db_pairs')
sys.argv = ['notebook']
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

SNAP_MAX = 4.0

con = duckdb.connect()
con.execute("""
CREATE VIEW g AS SELECT * FROM read_parquet(['work/game_tracking_2023.parquet',
  'work/game_tracking_2024.parquet', 'work/game_tracking_2025.parquet'])""")
df = con.execute(f"""
WITH pp AS (SELECT * FROM 'work/player_play.parquet' WHERE play_nullified_by_penalty = 'N'),
db AS (SELECT game_id, play_id, nfl_id AS db_id, team_abbr AS dteam, lined_up_position AS dpos
       FROM pp WHERE lined_up_position IN ('CB', 'SLOT_CB') AND coverage_assignment_at_snap = 'MAN'),
wr AS (SELECT game_id, play_id, nfl_id AS wr_id, team_abbr AS wteam, route_ran, cushion,
              separation_at_pass_forward AS off_sep, target::INT AS target
       FROM pp WHERE route_ran IS NOT NULL),
ev AS (SELECT game_id, play_id,
              min(time) FILTER (WHERE event = 'ball_snap') AS t_snap,
              min(time) FILTER (WHERE event = 'pass_forward') AS t_pf
       FROM g GROUP BY 1, 2 HAVING min(time) FILTER (WHERE event = 'pass_forward') IS NOT NULL),
pairs AS (SELECT * FROM db JOIN wr USING (game_id, play_id) JOIN ev USING (game_id, play_id) WHERE dteam <> wteam)
SELECT p.*, d0.x AS dx0, d0.y AS dy0, w0.x AS wx0, w0.y AS wy0,
       d1.x AS dx1, d1.y AS dy1, w1.x AS wx1, w1.y AS wy1
FROM pairs p
JOIN g d0 ON d0.game_id = p.game_id AND d0.play_id = p.play_id AND d0.nfl_id = p.db_id AND d0.time = p.t_snap
JOIN g w0 ON w0.game_id = p.game_id AND w0.play_id = p.play_id AND w0.nfl_id = p.wr_id AND w0.time = p.t_snap
JOIN g d1 ON d1.game_id = p.game_id AND d1.play_id = p.play_id AND d1.nfl_id = p.db_id AND d1.time = p.t_pf
JOIN g w1 ON w1.game_id = p.game_id AND w1.play_id = p.play_id AND w1.nfl_id = p.wr_id AND w1.time = p.t_pf
""").df()
df["dist_snap"] = np.hypot(df.dx0 - df.wx0, df.dy0 - df.wy0)
df["dist_pf"] = np.hypot(df.dx1 - df.wx1, df.dy1 - df.wy1)
print("man-coverage DB/receiver pairs with both tracked at snap and pass_forward:", len(df))

kept = df[df.dist_snap <= SNAP_MAX].copy()
print(f"kept (DB within {SNAP_MAX} yd of the receiver at the snap): {len(kept)}; "
      f"DBs: {kept.db_id.nunique()}; receivers: {kept.wr_id.nunique()}")
match = (kept.dist_pf - kept.off_sep).abs() < 0.15
print(f"check: DB-receiver distance equals the file's nearest-defender separation on {match.mean():.0%} of kept pairs "
      f"(on {(df[df.dist_snap > SNAP_MAX].pipe(lambda d: ((d.dist_pf - d.off_sep).abs() < 0.15).mean())):.0%} of the dropped ones)")

kept["exp"] = kept.groupby("route_ran").dist_pf.transform("mean")
kept["sep_allowed_over_exp"] = kept.dist_pf - kept.exp
kept.to_parquet(HERE / "work" / "db_pairs.parquet")

print("\nmean separation by route (yards):")
print(kept.groupby("route_ran").dist_pf.agg(["size", "mean"]).round(2).sort_values("size", ascending=False).to_string())

per = kept.groupby("db_id").agg(pairs=("dist_pf", "size"), sep_over_exp=("sep_allowed_over_exp", "mean")).reset_index()
print("\npairs per DB:", per.pairs.describe().round(1).to_dict())
print("DBs with >= 10 / 20 / 30 pairs:", [(per.pairs >= k).sum() for k in (10, 20, 30)])

# Split-half reliability of the outcome: odd vs even plays within each DB
kept = kept.sort_values(["db_id", "game_id", "play_id", "wr_id"])
kept["k"] = kept.groupby("db_id").cumcount()
for mn in (10, 20):
    ok = per[per.pairs >= mn].db_id
    h = kept[kept.db_id.isin(ok)].groupby(["db_id", kept.k % 2]).sep_allowed_over_exp.mean().unstack()
    r = h.corr(method="spearman").iloc[0, 1]
    print(f"split-half (odd vs even plays) reliability of separation allowed, DBs with >= {mn} pairs "
          f"(n={len(h)}): rho {r:.3f}, Spearman-Brown {2 * r / (1 + r):.3f}")

_log.stop()

## 17. Figures

Figures for the write-up, drawn from the saved result files. Each figure also writes the
values it plots to figures/<name>.csv, which is its text version.

Reads work/reliability.csv, work/overlap_results.csv, work/wr_matched_results.csv,
work/trench_results_original.csv, work/trench_results.csv and results/14_robustness_ol.txt.

In [ ]:
sys.argv = ['notebook']
import re
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = HERE / "figures"
OUT.mkdir(exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"      # validated pair (see dataviz validator): CVD and contrast pass
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#dcdcd8"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
})


def save(fig, name, data: pd.DataFrame):
    fig.savefig(OUT / f"{name}.png", dpi=160, bbox_inches="tight")
    data.to_csv(OUT / f"{name}.csv", index=False)
    plt.close(fig)


# ---------------------------------------------------------------- Figure 1: reliability
rel = pd.read_csv(HERE / "work" / "reliability.csv")
rel = rel[(rel.side == "OL") & (rel.drill == "PASS_RUSH_DROP") | (rel.side == "DL") & (rel.drill == "PASS_RUSH")]
NAMES = {
    "speed_0p5s": "Speed 0.5 s after start", "burst_dist_1s": "Distance in 1st second after start",
    "peak_accel_1s": "Peak acceleration, 1st second", "turn_rate": "Turn rate",
    "peak_speed": "Peak speed", "best_net_1s": "Best distance in any 1 s",
    "best_gain_0p5s": "Best speed gain in any 0.5 s",
}
ORDER = ["speed_0p5s", "burst_dist_1s", "peak_accel_1s", "turn_rate",       # need a detected start
         "peak_speed", "best_net_1s", "best_gain_0p5s"]                      # do not
p = rel[rel.feature.isin(ORDER)].pivot(index="feature", columns="side", values="attempt_corr").loc[ORDER]
fig, ax = plt.subplots(figsize=(8.6, 5.2))
y = np.arange(len(ORDER))[::-1].astype(float)
y[4:] -= 0.6   # gap between the two groups
h = 0.34
for off, side, col in [(h / 2, "DL", ORANGE), (-h / 2, "OL", BLUE)]:
    ax.barh(y + off, p[side], height=h, color=col, label=f"{'Defensive' if side == 'DL' else 'Offensive'} line")
    for yy, v in zip(y + off, p[side]):
        ax.text(v + 0.012, yy, f"{v:.2f}", va="center", fontsize=9.5, color=INK)
ax.set_yticks(y)
ax.set_yticklabels([NAMES[f] for f in ORDER])
ax.axvline(0.6, color=MUTED, ls="--", lw=1)
ax.text(0.605, y.max() + 0.62, "0.6: level we aimed for", fontsize=9, color=MUTED, va="bottom")
ax.set_ylim(y.min() - 0.6, y.max() + 1.25)
ax.set_xlim(0, 0.85)
ax.set_xlabel("Correlation between an athlete's first and second attempt (Spearman)")
ax.text(0.0, y[0] + 0.62, "Needs a detected start frame", fontsize=10, color=INK, fontweight="bold")
ax.text(0.0, y[4] + 0.62, "Does not need a start frame", fontsize=10, color=INK, fontweight="bold")
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower right")
ax.set_title("Measures that need a start frame barely repeat", loc="left", fontsize=13, fontweight="bold")
save(fig, "fig1_reliability", p.reset_index().rename(columns={"OL": "OL_pass_rush_drop", "DL": "DL_pass_rush"}))

# ---------------------------------------------------------------- Figure 2: overlap
ov = pd.read_csv(HERE / "work" / "overlap_results.csv")
LAB = {"speed_lost": "speed lost", "reaccel_s": "time to re-accelerate", "break_angle": "break angle",
       "entry_speed": "entry speed", "peak_speed": "peak speed", "best_net_1s": "best distance in 1 s",
       "best_gain_0p5s": "best speed gain in 0.5 s", "peak_speed_s": "peak speed (file's speed)"}
ov["label"] = ov.group.str.replace(" drills", "").str.replace("WR sharp cuts", "WR cuts") + ": " + ov.feature.map(LAB)
ov = ov.sort_values("cv_r2_stopwatch")
fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.barh(ov.label, ov.cv_r2_stopwatch, height=0.62, color=BLUE)
for yy, v in zip(range(len(ov)), ov.cv_r2_stopwatch):
    ax.text(max(v, 0) + 0.01, yy, f"{v:.2f}", va="center", fontsize=9.5, color=INK)
ax.axvline(0, color=MUTED, lw=1)
ax.set_xlim(-0.12, 0.8)
ax.set_xlabel("Share of the feature predicted by 40 time, 10-yard split, jumps, weight and height (cross-validated R-squared)")
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
ax.set_title("Only defensive-line speed measures mostly repeat the stopwatch", loc="left", fontsize=13, fontweight="bold")
save(fig, "fig2_overlap", ov[["group", "feature", "n", "cv_r2_stopwatch", "rho_with_forty"]].iloc[::-1])

# ---------------------------------------------------------------- Figure 3: every test
fam = []
w = pd.read_csv(HERE / "work" / "wr_matched_results.csv").assign(family="Receivers by route (19)")
w["name"] = w.route + " " + w.feature
t1 = pd.read_csv(HERE / "work" / "trench_results_original.csv").assign(family="Linemen, first pass (20)")
t2 = pd.read_csv(HERE / "work" / "trench_results.csv").assign(family="Linemen, second pass (12)")
for d in (t1, t2):
    d["name"] = d.side + " " + d.feature + " vs " + d.outcome.str.replace("_over_exp", "").str.replace("_over_pos", "")
al = pd.concat([w, t1, t2], ignore_index=True)
al["excludes_zero"] = (al.ci_lo > 0) | (al.ci_hi < 0)
fig, ax = plt.subplots(figsize=(8.6, 6.4))
ypos, labels, ylab = 0, [], []
FAMS = ["Receivers by route (19)", "Linemen, first pass (20)", "Linemen, second pass (12)"]
for f in FAMS:
    d = al[al.family == f].sort_values("partial_rho").reset_index(drop=True)
    ys = ypos + np.arange(len(d))
    for yy, (_, r) in zip(ys, d.iterrows()):
        col = ORANGE if r.excludes_zero else BLUE
        if not np.isnan(r.ci_lo):
            ax.plot([r.ci_lo, r.ci_hi], [yy, yy], color=col, lw=1.6, alpha=0.55 if not r.excludes_zero else 1)
        ax.plot(r.partial_rho, yy, "o", color=col, ms=5.5, mec="white", mew=1)
        if r.excludes_zero:
            ax.text(r.ci_hi + 0.02, yy, f"{r['name']}: p {r.p_boot:.3f}, adjusted {r.p_bh:.2f}", fontsize=8.5, va="center", color=INK)
    ax.text(-0.98, ys.max() + 1.6, f, fontsize=10, fontweight="bold", color=INK)
    ypos = ys.max() + 4
ax.axvline(0, color=MUTED, lw=1)
ax.set_xlim(-1, 1)
ax.set_ylim(-1.5, ypos - 1.5)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_xlabel("Partial rank correlation with the NFL outcome, with 95% interval")
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
n_ex = int(al.excludes_zero.sum())
ax.text(0.98, 0.02, "Blue: interval includes zero.  Orange: interval misses zero.", transform=ax.transAxes, ha="right", fontsize=9, color=MUTED)
ax.set_title(f"{len(al)} movement tests: {n_ex} intervals miss zero, none survive correction", loc="left", fontsize=13, fontweight="bold")
save(fig, "fig3_all_tests", al[["family", "name", "n", "partial_rho", "ci_lo", "ci_hi", "p_boot", "p_bh", "excludes_zero"]])

# ---------------------------------------------------------------- Figure 4: robustness of the OL lead
log = (HERE / "results" / "14_robustness_ol.txt").read_text()
num = r"np\.float64\((-?[\d.]+)\)"
pat = re.compile(r"^\s{3}(.+?)\s+n,rho,ci:\s*\[(\d+), " + num + ", " + num + ", " + num + r"\]", re.M)
rows = [(m.group(1).strip(), int(m.group(2)), *map(float, m.groups()[2:])) for m in pat.finditer(log)]
pat2 = re.compile(r"^\s{3}(>= +\d+ plays)\s+n,rho,ci:\s*\[(\d+), " + num + ", " + num + ", " + num + r"\]", re.M)
rows += [(m.group(1).replace("  ", " "), int(m.group(2)), *map(float, m.groups()[2:])) for m in pat2.finditer(log)]
m3 = re.search(r"team-mean removed .*?\[(\d+), " + num + ", " + num + ", " + num + r"\]", log)
rows.append(("team-mean removed", int(m3.group(1)), *map(float, m3.groups()[1:])))
rb = pd.DataFrame(rows, columns=["check", "n", "rho", "ci_lo", "ci_hi"]).drop_duplicates("check")
KEEP = {
    "ol_pressure_rate_over_exp": "Original: pressure rate over expected",
    "ol_pressure_prob_over_exp": "Pressure probability over expected",
    "raw_pressure": "Raw pressure rate",
    "sack_rate": "Sack rate (expected +? no: lower is better)",
    "mean_ttp (>=20 pressured plays)": "Time to pressure (higher is better)",
    "team-mean removed": "Team average removed",
    "best_net_1s in PASS_PRO_MIRROR": "Mirror drill only",
    "best_net_1s in PASS_RUSH_DROP": "Pass-rush-drop drill only",
    "best_net_1s in WAVE_SLIDE": "Wave-slide drill only",
    "best_path_1s_s (file's own speed)": "Distance from file's own speed",
    ">= 200 plays": "At least 200 pass-block plays",
    ">= 300 plays": "At least 300 pass-block plays",
}
KEEP["sack_rate"] = "Sack rate"
rb = rb[rb.check.isin(KEEP)].copy()
rb["label"] = rb.check.map(KEEP)
rb = rb.set_index("check").loc[list(KEEP)].reset_index()
fig, ax = plt.subplots(figsize=(8.6, 5.6))
yy = np.arange(len(rb))[::-1]
for y_, (_, r) in zip(yy, rb.iterrows()):
    col = BLUE if r.ci_hi < 0 else MUTED
    ax.plot([r.ci_lo, r.ci_hi], [y_, y_], color=col, lw=1.8)
    ax.plot(r.rho, y_, "o", color=BLUE if r.check == "ol_pressure_rate_over_exp" else col, ms=7, mec="white", mew=1.2)
    ax.text(0.5, y_, f"{r.rho:+.2f}", va="center", fontsize=9.5, color=INK)
ax.set_yticks(yy)
ax.set_yticklabels(rb.label)
ax.axvline(0, color=MUTED, lw=1)
ax.set_xlim(-0.6, 0.62)
ax.set_xlabel("Correlation of best distance in 1 s with the outcome, with 95% interval")
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
ax.set_title("The offensive-line lead fades under other definitions", loc="left", fontsize=13, fontweight="bold")
fig.text(0.02, -0.03, "Negative means more distance goes with less pressure allowed, except time to pressure, where positive is better.",
         fontsize=9, color=MUTED)
save(fig, "fig4_ol_robustness", rb[["label", "n", "rho", "ci_lo", "ci_hi"]])
print("wrote", sorted(p.name for p in OUT.iterdir()))
print("tests:", len(al), "| excluding zero:", n_ex)
print(al[al.excludes_zero][["family", "name", "partial_rho", "ci_lo", "ci_hi", "p_boot", "p_bh"]].round(3).to_string(index=False))


from IPython.display import Image, display
for _f in ["fig1_reliability", "fig2_overlap", "fig3_all_tests", "fig4_ol_robustness"]:
    display(Image(str(HERE / "figures" / f"{_f}.png")))

## Reading the results

- Sections 4, 5, 9 and 12 hold the tests. None of the 51 partial-correlation tests survives correction for multiple
  comparisons.
- Sections 10, 11 and 13 show that start-based first-step measures barely repeat between attempts, and that some
  measures mostly repeat the 40-yard dash.
- Sections 14 and 15 check the one offensive-line lead; it does not hold up.
- Section 16 shows why the defensive-back question cannot be answered from this data.

The full write-up explains the method, limits and what teams can take from this.